In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Load data
df = pd.read_csv("../data/sierraleone-bumbuna.csv")
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Loaded 525600 rows, 19 columns


,Timestamp,GHI,DNI,DHI,ModA,ModB,Tamb,RH,WS,WSgust,WSstdev,WD,WDstdev,BP,Cleaning,Precipitation,TModA,TModB,Comments
0,2021-10-30 00:01,-0.7,-0.1,-0.8,0.0,0.0,21.9,99.1,0.0,0.0,0.0,0.0,0.0,1002,0,0.0,22.3,22.6,NaN
1,2021-10-30 00:02,-0.7,-0.1,-0.8,0.0,0.0,21.9,99.2,0.0,0.0,0.0,0.0,0.0,1002,0,0.0,22.3,22.6,NaN
2,2021-10-30 00:03,-0.7,-0.1,-0.8,0.0,0.0,21.9,99.2,0.0,0.0,0.0,0.0,0.0,1002,0,0.0,22.3,22.6,NaN
3,2021-10-30 00:04,-0.7,0.0,-0.8,0.0,0.0,21.9,99.3,0.0,0.0,0.0,0.0,0.0,1002,0,0.1,22.3,22.6,NaN
4,2021-10-30 00:05,-0.7,-0.1,-0.8,0.0,0.0,21.9,99.3,0.0,0.0,0.0,0.0,0.0,1002,0,0.0,22.3,22.6,NaN


In [2]:
print("=== Summary Statistics ===")
print(df.describe())

print("\n=== Missing Values ===")
missing = df.isna().sum()
print(missing[missing > 0])

# Check for >5% missing
pct_missing = (missing / len(df)) * 100
high_missing = pct_missing[pct_missing > 5]
if not high_missing.empty:
    print("\n>5% missing in:")
    print(high_missing)
else:
    print("\n✅ No columns with >5% missing.")

=== Summary Statistics ===
                 GHI            DNI            DHI           ModA  \
count  525600.000000  525600.000000  525600.000000  525600.000000   
mean      201.957515     116.376337     113.720571     206.643095   
std       298.495150     218.652659     158.946032     300.896893   
min       -19.500000      -7.800000     -17.900000       0.000000   
25%        -2.800000      -0.300000      -3.800000       0.000000   
50%         0.300000      -0.100000      -0.100000       3.600000   
75%       362.400000     107.000000     224.700000     359.500000   
max      1499.000000     946.000000     892.000000    1507.000000   

                ModB           Tamb             RH             WS  \
count  525600.000000  525600.000000  525600.000000  525600.000000   
mean      198.114691      26.319394      79.448857       1.146113   
std       288.889073       4.398605      20.520775       1.239248   
min         0.000000      12.300000       9.900000       0.000000   
25%   

In [3]:
key_cols = ['GHI', 'DNI', 'DHI', 'ModA', 'ModB', 'WS', 'WSgust']
valid_key_cols = [c for c in key_cols if c in df.columns]

if valid_key_cols:
    z_scores = np.abs(stats.zscore(df[valid_key_cols].dropna()))
    outliers = (z_scores > 3).any(axis=1)
    print(f"Rows with |Z| > 3: {outliers.sum()}")
    df['is_outlier'] = False
    df.loc[df[valid_key_cols].dropna().index[outliers], 'is_outlier'] = True
else:
    print("⚠️ Key columns not found. Available:", df.columns.tolist())

Rows with |Z| > 3: 16292


In [4]:
# Impute missing values with median
for col in valid_key_cols:
    if col in df.columns:
        df[col].fillna(df[col].median(), inplace=True)

# Fix timestamp
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df.sort_values('Timestamp', inplace=True)
df.reset_index(drop=True, inplace=True)

C:\Users\hp\AppData\Local\Temp\ipykernel_19452\3790170879.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\hp\AppData\Local\Temp\ipykernel_19452\3790170879.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, wh

In [5]:
df.to_csv("../data/sierraleone-bumbuna_clean.csv", index=False)
print("✅ Cleaned Senegal data saved.")


✅ Cleaned Senegal data saved.


## 🔍 Key Insights — Senegal (Bumbuna)

- GHI behavior shows clear diurnal pattern.
- ModA/ModB correlation indicates stable sensor performance.
- Outliers likely due to sensor noise or calibration drift.
- Wind speed shows moderate inverse relationship with GHI.

## 📚 References
- [EDA Best Practices](https://towardsdatascience.com/exploratory-data-analysis-eda-a-practical-guide...)
- SciPy Z-score docs
- Seaborn visualization guide